# 11 - Sentinel-2 Per-Cell Features and Image Patches

Turns the five yearly Sentinel-2 mosaics (2020-2024, 10 m, 9 bands: B2, B3, B4, B8, B11, B12, NDVI, EVI, NDMI) into two products
keyed to the 1 km analysis grid:

1. `datasets/processed/features/sentinel2_cell_stats.csv`: per-cell summary statistics (tabular, one row per `grid_id`).
2. `datasets/processed/sentinel2_patches/patches_<year>.npy`: one 25 x 25 image patch per cell and year (4 x 4 block means of
   the 100 x 100 pixels inside the cell; 9 bands plus a validity channel), for convolutional models.

The grid is aligned to the mosaic: the origin of the 1 km grid sits at whole-pixel offsets of the 10 m raster, so every cell is
exactly a 100 x 100 pixel window and no resampling or polygon rasterisation is needed. Pixels without data (outside the province,
masked clouds) are NaN in the mosaics and are ignored by every statistic. EVI has extreme outliers where the denominator is near
zero, so it is clipped to [-1, 2] before aggregation.

In [1]:
import time
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").exists())
S2_DIR = PROJECT_ROOT / "datasets" / "processed" / "sentinel2"
FEATURES_DIR = PROJECT_ROOT / "datasets" / "processed" / "features"
PATCH_DIR = PROJECT_ROOT / "datasets" / "processed" / "sentinel2_patches"
PATCH_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024]
BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "EVI", "NDMI"]
CELL_PX = 100          # 1 km / 10 m
POOL = 4               # 100 px -> 25 px patches
GRID_MINX, GRID_MINY = 511000.0, 9222000.0   # south-west corner of the 1 km grid (EPSG:32748); grid row 0 is the southern edge

cells = pd.read_csv(FEATURES_DIR / "tabular_features.csv", usecols=["grid_id", "row", "col"])
print(len(cells), "cells;", cells["row"].max() + 1, "grid rows x", cells["col"].max() + 1, "grid cols")

# the (row, col) indices must agree with the cell polygons, otherwise the windows below would be wrong
grid = gpd.read_file(PROJECT_ROOT / "datasets" / "processed" / "grid" / "grid_1km.gpkg").merge(cells[["grid_id"]], on="grid_id")
b = grid.geometry.bounds
assert np.allclose((b["minx"] - GRID_MINX) / 1000, grid["col"]) and np.allclose((b["miny"] - GRID_MINY) / 1000, grid["row"])

9333 cells; 136 grid rows x 186 grid cols


In [2]:
with rasterio.open(S2_DIR / "sentinel2_banten_2020.tif") as src:
    MOSAIC_LEFT, MOSAIC_TOP, MOSAIC_RES = src.transform.c, src.transform.f, src.transform.a
    assert src.descriptions == tuple(BANDS), src.descriptions
    print("mosaic:", src.width, "x", src.height, "px at", MOSAIC_RES, "m")

def cell_window(row, col, height, width):
    """Pixel window of a grid cell: (r0, r1, c0, c1) clipped to the mosaic, plus the unclipped (r0, c0) of the full 100 x 100 cell."""
    r0 = int(round((MOSAIC_TOP - (GRID_MINY + (row + 1) * 1000.0)) / MOSAIC_RES))
    c0 = int(round((GRID_MINX + col * 1000.0 - MOSAIC_LEFT) / MOSAIC_RES))
    return max(r0, 0), min(r0 + CELL_PX, height), max(c0, 0), min(c0 + CELL_PX, width), r0, c0

# the grid origin must sit on whole mosaic pixels, otherwise "100 x 100 pixels per cell" would not hold
assert abs((GRID_MINX - MOSAIC_LEFT) / MOSAIC_RES - round((GRID_MINX - MOSAIC_LEFT) / MOSAIC_RES)) < 1e-6
print("example windows (r0, r1, c0, c1, full_r0, full_c0):", [cell_window(r, c, 13574, 18588) for r, c in [(0, 0), (70, 100), (135, 185)]])

mosaic: 18588 x 13574 px at 10.0 m
example windows (r0, r1, c0, c1, full_r0, full_c0): [(13489, 13574, 0, 97, 13489, -3), (6489, 6589, 9997, 10097, 6489, 9997), (0, 89, 18497, 18588, -11, 18497)]


## Extract (one pass per year; one strip of 100 mosaic rows per grid row)

In [3]:
def pool_patch(block, dr, dc):
    """block: (9, h, w), NaN = no data, pasted at offset (dr, dc) of the full cell -> (10, 25, 25): 9 band means over 4 x 4 pixels + valid share."""
    full = np.full((len(BANDS), CELL_PX, CELL_PX), np.nan, dtype=np.float32)
    full[:, dr: dr + block.shape[1], dc: dc + block.shape[2]] = block
    g = full.reshape(len(BANDS), CELL_PX // POOL, POOL, CELL_PX // POOL, POOL)
    with warnings.catch_warnings():                          # 4 x 4 blocks without any valid pixel stay NaN -> 0
        warnings.simplefilter("ignore", RuntimeWarning)
        mean = np.nanmean(g, axis=(2, 4))
    frac = np.isfinite(g[3]).mean(axis=(1, 3))               # B8 decides validity (all bands share one mask)
    return np.concatenate([np.nan_to_num(mean, nan=0.0), frac[None]], axis=0)

def cell_stats(block):
    valid = np.isfinite(block[3])
    n = int(valid.sum())
    out = {"valid_frac": n / (CELL_PX * CELL_PX)}
    if n == 0:
        for b_ in BANDS:
            out[f"{b_}_mean"] = out[f"{b_}_std"] = np.nan
        out["NDVI_p10"] = out["NDVI_p50"] = out["NDVI_p90"] = np.nan
        return out
    for i, b_ in enumerate(BANDS):
        v = block[i][valid]
        out[f"{b_}_mean"], out[f"{b_}_std"] = float(v.mean()), float(v.std())
    out["NDVI_p10"], out["NDVI_p50"], out["NDVI_p90"] = map(float, np.percentile(block[6][valid], [10, 50, 90]))
    return out

EMPTY = np.full((len(BANDS), 1, 1), np.nan, dtype=np.float32)
long_rows = []
for year in YEARS:
    t0 = time.time()
    patches = np.zeros((len(cells), len(BANDS) + 1, CELL_PX // POOL, CELL_PX // POOL), dtype=np.float16)
    with rasterio.open(S2_DIR / f"sentinel2_banten_{year}.tif") as src:
        H, W = src.height, src.width
        for grid_row, grp in cells.groupby("row"):
            r0, r1, _, _, full_r0, _ = cell_window(grid_row, 0, H, W)
            strip = None
            if r1 > r0:
                strip = src.read(window=Window(0, r0, W, r1 - r0)).astype(np.float32)
                strip[7] = np.clip(strip[7], -1.0, 2.0)
            for idx, col in zip(grp.index, grp["col"]):
                _, _, c0, c1, _, full_c0 = cell_window(grid_row, col, H, W)
                block = strip[:, :, c0:c1] if strip is not None and c1 > c0 else EMPTY
                long_rows.append({"grid_id": cells.at[idx, "grid_id"], "year": year, **cell_stats(block)})
                patches[idx] = pool_patch(block, max(r0 - full_r0, 0), max(c0 - full_c0, 0)).astype(np.float16)
    np.save(PATCH_DIR / f"patches_{year}.npy", patches)
    print(f"{year}: done in {time.time() - t0:.0f} s -> patches_{year}.npy {patches.shape}", flush=True)

long = pd.DataFrame(long_rows)
long.to_csv(FEATURES_DIR / "sentinel2_cell_stats_long.csv", index=False)
long.shape

2020: done in 23 s -> patches_2020.npy (9333, 10, 25, 25)


2021: done in 28 s -> patches_2021.npy (9333, 10, 25, 25)


2022: done in 22 s -> patches_2022.npy (9333, 10, 25, 25)


2023: done in 24 s -> patches_2023.npy (9333, 10, 25, 25)


2024: done in 25 s -> patches_2024.npy (9333, 10, 25, 25)


(46665, 24)

## Multi-year summary
One row per cell: the five-year mean of every statistic plus the year-to-year variability of NDVI and NDMI. The
dashboard joins it to the fact table on `grid_id`; m14 tests what it adds to the physical drivers.

In [4]:
stat_cols = [c for c in long.columns if c not in ("grid_id", "year")]
summary = long.groupby("grid_id")[stat_cols].mean().add_prefix("s2_")
iav = long.pivot(index="grid_id", columns="year", values=["NDVI_mean", "NDMI_mean"])
summary["s2_NDVI_mean_iav"] = iav["NDVI_mean"].std(axis=1)      # inter-annual standard deviation of the cell's mean NDVI
summary["s2_NDMI_mean_iav"] = iav["NDMI_mean"].std(axis=1)
summary = cells[["grid_id"]].merge(summary.reset_index(), on="grid_id", how="left")
summary.to_csv(FEATURES_DIR / "sentinel2_cell_stats.csv", index=False)
print(summary.shape, "| cells with no Sentinel-2 pixels in any year:", int(summary["s2_valid_frac"].fillna(0).eq(0).sum()))
summary.describe().T[["count", "mean", "std", "min", "max"]].round(3)

(9333, 25) | cells with no Sentinel-2 pixels in any year: 0


,count,mean,std,min,max
s2_valid_frac,9333.0,0.934,0.103,0.113,1.000
s2_B2_mean,9333.0,0.052,0.018,0.022,0.176
s2_B2_std,9333.0,0.016,0.010,0.003,0.082
s2_B3_mean,9333.0,0.072,0.018,0.030,0.206
s2_B3_std,9333.0,0.018,0.009,0.003,0.092
s2_B4_mean,9333.0,0.058,0.026,0.020,0.207
s2_B4_std,9333.0,0.024,0.012,0.002,0.126
s2_B8_mean,9333.0,0.289,0.046,0.017,0.406
s2_B8_std,9333.0,0.052,0.011,0.002,0.127
s2_B11_mean,9333.0,0.190,0.030,0.011,0.330


In [5]:
# Sanity check against an independent layer: NDVI should be high on tree cover / cropland and low on built-up land.
tab = pd.read_csv(FEATURES_DIR / "tabular_features.csv", usecols=["grid_id", "landcover_majority", "cropland_pct"])
chk = tab.merge(summary, on="grid_id")
chk.groupby("landcover_majority")[["s2_NDVI_mean", "s2_NDMI_mean", "s2_B11_mean", "s2_valid_frac"]].mean().round(3).assign(cells=chk.groupby("landcover_majority").size())

,s2_NDVI_mean,s2_NDMI_mean,s2_B11_mean,s2_valid_frac,cells
landcover_majority,,,,,
10.0,0.731,0.242,0.186,0.937,6989
30.0,0.451,0.055,0.239,0.889,76
40.0,0.516,0.151,0.195,0.932,1282
50.0,0.329,-0.018,0.226,0.967,772
60.0,0.176,-0.037,0.253,0.908,2
80.0,0.207,0.218,0.109,0.769,193
90.0,0.354,0.164,0.144,0.913,12
95.0,0.797,0.452,0.108,0.678,7
